<!-- NB_DOC_INTRO_V1 -->
# ML — Multi-armed bandits (UCB, Thompson Sampling)

> 📚 **Doc thematique** : [docs/03_ML.md](docs/03_ML.md) (Machine Learning classique)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Multi-armed bandits** = cas restreint du RL : a chaque tour, choisir une action (bras) parmi N, recevoir une recompense. Objectif : maximiser la **recompense cumulee** = balance **exploration** (essayer de nouveaux bras) vs **exploitation** (jouer le bras meilleur connu).

Algos couverts : ε-greedy, UCB (Upper Confidence Bound), Thompson Sampling (bayesien). Comparatif sur un environnement simule (10 bras). Cas reels : A/B testing online, recommandation, allocation budgetaire pub.

⚠️ **C'est PAS du RL complet** (pas d'etat, pas de sequence). Pour vrai RL : voir gymnasium + stable-baselines3.

## Plan

1. Setup bandit environment
2. Strategie aleatoire (baseline)
3. ε-greedy
4. UCB (Upper Confidence Bound)
5. Thompson Sampling (bayesien)
6. Comparatif regret cumule
7. Pieges et anti-patterns
8. References


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

## 1. Setup — environnement bandit

In [ ]:
# 10 bras, chacun a une vraie probabilite de recompense (cachee) entre 0 et 1
N_ARMS = 10
N_ROUNDS = 5000
true_probs = np.random.uniform(0.1, 0.9, N_ARMS)
print(f"Vraies probas (cachees) : {true_probs.round(3)}")
print(f"Meilleur bras (oracle)  : {np.argmax(true_probs)} (proba={true_probs.max():.3f})")

def pull(arm: int) -> int:
    """Tirer un bras : recompense binaire 0/1 selon Bernoulli(p[arm])."""
    return int(np.random.rand() < true_probs[arm])

## 2. Strategie aleatoire (baseline)

In [ ]:
def random_strategy(n_rounds, n_arms):
    rewards = []
    choices = []
    for _ in range(n_rounds):
        arm = np.random.randint(n_arms)
        reward = pull(arm)
        rewards.append(reward); choices.append(arm)
    return np.array(rewards), np.array(choices)

np.random.seed(SEED)
r_random, _ = random_strategy(N_ROUNDS, N_ARMS)
print(f"Random : recompense totale = {r_random.sum()}")

## 3. ε-greedy

Avec proba `ε` : exploration aleatoire. Avec proba `1-ε` : exploitation (meilleur bras connu).

In [ ]:
def epsilon_greedy(n_rounds, n_arms, epsilon=0.1):
    counts = np.zeros(n_arms)
    means = np.zeros(n_arms)
    rewards = []
    for t in range(n_rounds):
        if np.random.rand() < epsilon:
            arm = np.random.randint(n_arms)
        else:
            arm = int(np.argmax(means))
        reward = pull(arm)
        counts[arm] += 1
        means[arm] += (reward - means[arm]) / counts[arm]   # mise a jour incrementale
        rewards.append(reward)
    return np.array(rewards)

np.random.seed(SEED)
r_eps01 = epsilon_greedy(N_ROUNDS, N_ARMS, epsilon=0.1)
np.random.seed(SEED)
r_eps05 = epsilon_greedy(N_ROUNDS, N_ARMS, epsilon=0.05)
print(f"ε=0.10 : total = {r_eps01.sum()}")
print(f"ε=0.05 : total = {r_eps05.sum()}")

## 4. UCB1 (Upper Confidence Bound)

Choisir le bras qui maximise : `mean[arm] + sqrt(2 * ln(t) / counts[arm])`. Penalise les bras sous-explores (uncertainty bonus).

In [ ]:
def ucb(n_rounds, n_arms):
    counts = np.zeros(n_arms)
    means = np.zeros(n_arms)
    rewards = []

    # Initialisation : pull chaque bras 1 fois
    for arm in range(n_arms):
        r = pull(arm)
        counts[arm] = 1
        means[arm] = r
        rewards.append(r)

    for t in range(n_arms, n_rounds):
        # UCB1 score
        ucb_scores = means + np.sqrt(2 * np.log(t + 1) / counts)
        arm = int(np.argmax(ucb_scores))
        reward = pull(arm)
        counts[arm] += 1
        means[arm] += (reward - means[arm]) / counts[arm]
        rewards.append(reward)
    return np.array(rewards)

np.random.seed(SEED)
r_ucb = ucb(N_ROUNDS, N_ARMS)
print(f"UCB1 : total = {r_ucb.sum()}")

## 5. Thompson Sampling (bayesien)

Maintient une distribution Beta(α, β) par bras. A chaque tour : sample de chaque Beta, joue le bras avec le sample le plus haut. Convergence theorique optimale en log.

In [ ]:
def thompson_sampling(n_rounds, n_arms):
    # Beta(1, 1) = uniforme = ignorance totale
    alpha = np.ones(n_arms)
    beta = np.ones(n_arms)
    rewards = []
    for t in range(n_rounds):
        samples = np.random.beta(alpha, beta)
        arm = int(np.argmax(samples))
        reward = pull(arm)
        if reward == 1:
            alpha[arm] += 1
        else:
            beta[arm] += 1
        rewards.append(reward)
    return np.array(rewards)

np.random.seed(SEED)
r_thomp = thompson_sampling(N_ROUNDS, N_ARMS)
print(f"Thompson Sampling : total = {r_thomp.sum()}")
print(f"Oracle (max theorique) : {true_probs.max() * N_ROUNDS:.0f}")

## 6. Comparatif — regret cumule

Le **regret** = (oracle - reward) cumule au cours du temps. Plus c'est plat (proche de 0), mieux c'est.

In [ ]:
oracle = true_probs.max()

def regret_cumul(rewards):
    return np.cumsum(oracle - rewards)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(regret_cumul(r_random), label=f"Random (final {regret_cumul(r_random)[-1]:.0f})")
ax.plot(regret_cumul(r_eps01), label=f"ε-greedy 0.1 ({regret_cumul(r_eps01)[-1]:.0f})")
ax.plot(regret_cumul(r_eps05), label=f"ε-greedy 0.05 ({regret_cumul(r_eps05)[-1]:.0f})")
ax.plot(regret_cumul(r_ucb), label=f"UCB1 ({regret_cumul(r_ucb)[-1]:.0f})")
ax.plot(regret_cumul(r_thomp), label=f"Thompson ({regret_cumul(r_thomp)[-1]:.0f})")
ax.set(xlabel="Round", ylabel="Regret cumule", title="Comparaison regret cumule (plus bas = mieux)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| ε constant haut (0.2+) | Decay : ε = 1/sqrt(t) → moins exploration en fin |
| UCB1 sans burn-in initial | Initialiser counts >= 1 par bras |
| Confondre bandits et A/B test classique | Bandits = adaptif, A/B = split fige a l'avance |
| Bandit sans context (features) | **Contextual bandits** (LinUCB, Thompson) capturent le profil user |
| Reporter recompense totale sans regret | Regret = mesure absolue de qualite |
| Pas tester sur plusieurs seeds | Bandits stochastiques → moyenner sur N runs |
| Confondre bandits et RL complet | RL : etat + sequence. Bandits : sans etat |


## References

### Documentation
- *Bandit Algorithms* (Lattimore & Szepesvari) — gratuit : https://tor-lattimore.com/downloads/book/book.pdf
- Sutton & Barto — *Reinforcement Learning: An Introduction* : http://incompleteideas.net/book/the-book.html
- scikit-bandit : https://github.com/spotify/banditml

### Voir aussi
- [DS_Bayesian.ipynb](DS_Bayesian.ipynb)
- [DS_Causal_Inference.ipynb](DS_Causal_Inference.ipynb)
